In [1]:
print("hello")

hello


In [5]:

# STEP 1: DATA ACQUISITION

import pandas as pd
from pathlib import Path

# Dataset Path (Change this to your folder)
DATA_PATH = Path(
    "/Users/dishitasingh/Desktop/Drexel/INFO442/INFO442-project-main/INFO442-project/Kaggle_dataset"
)

# Rating Files
rating_files = [
    DATA_PATH / "combined_data_1.txt",
    DATA_PATH / "combined_data_2.txt",
    DATA_PATH / "combined_data_3.txt",
    DATA_PATH / "combined_data_4.txt"
]

print("Checking rating files...\n")

for file in rating_files:
    if file.exists():
        size_gb = file.stat().st_size / (1024**3)
        print(f"✓ {file.name} ({size_gb:.2f} GB)")
    else:
        print(f"✗ {file.name} NOT FOUND")


# Load Movie Titles
movie_file = DATA_PATH / "movie_titles.csv"
movie_records = []
with open(movie_file, "r", encoding="latin-1") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        movie_id, year, title = line.split(",", 2)

        movie_records.append({
            "movie_id": int(movie_id),
            "year": pd.to_numeric(year, errors="coerce"),
            "title": title
        })
movies = pd.DataFrame(movie_records)
movies["year"] = movies["year"].astype("Int64")


# Display Movie Information
print("\nMovie Titles Loaded Successfully!\n")
print(movies.head())

print("\nMovie Dataset Shape:")
print(movies.shape)


Checking rating files...

✓ combined_data_1.txt (0.46 GB)
✓ combined_data_2.txt (0.52 GB)
✓ combined_data_3.txt (0.43 GB)
✓ combined_data_4.txt (0.51 GB)

Movie Titles Loaded Successfully!

   movie_id  year                         title
0         1  2003               Dinosaur Planet
1         2  2004    Isle of Man TT 2004 Review
2         3  1997                     Character
3         4  1994  Paula Abdul's Get Up & Dance
4         5  2004      The Rise and Fall of ECW

Movie Dataset Shape:
(17770, 3)


In [ ]:
# STEP 2: Parse and Merge the Four Rating Files
# Output: ratings.csv

output_file = DATA_PATH / "ratings.csv"

# Create output file with header
with open(output_file, "w") as out:
    out.write("user_id,movie_id,rating,date\n")

for file in rating_files:
    print(f"Processing {file.name}...")
    current_movie = None
    rows = []

    with open(file, "r") as f:
        for line in f:
            line = line.strip()

            # Movie ID line
            if line.endswith(":"):
                current_movie = int(line[:-1])

            # Rating line
            else:
                user_id, rating, date = line.split(",")
                rows.append([
                    int(user_id),
                    current_movie,
                    int(rating),
                    date
                ])

            # Save every 1 million rows
            if len(rows) >= 1_000_000:
                pd.DataFrame(
                    rows,
                    columns=["user_id", "movie_id", "rating", "date"]
                ).to_csv(
                    output_file,
                    mode="a",
                    header=False,
                    index=False
                )
                print(f"  Saved {len(rows):,} rows")
                rows = []

    # Save remaining rows
    if rows:
        pd.DataFrame(
            rows,
            columns=["user_id", "movie_id", "rating", "date"]
        ).to_csv(
            output_file,
            mode="a",
            header=False,
            index=False
        )
        print(f"  Saved final {len(rows):,} rows")

print("\nAll rating files merged successfully!")
print(f"Saved to: {output_file}")


Processing combined_data_1.txt...
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved final 53,764 rows
Processing combined_data_2.txt...
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1

In [7]:
ratings = pd.read_csv(output_file)

print(ratings.head())
print(ratings.shape)

   user_id  movie_id  rating        date
0  1488844         1       3  2005-09-06
1   822109         1       5  2005-05-13
2   885013         1       4  2005-10-19
3    30878         1       4  2005-12-26
4   823519         1       3  2004-05-03
(100480507, 4)


In [9]:
# STEP 3: Clean the Data

# 1. Convert data types
ratings["user_id"] = pd.to_numeric(
    ratings["user_id"],
    errors="coerce"
)
ratings["movie_id"] = pd.to_numeric(
    ratings["movie_id"],
    errors="coerce"
)
ratings["rating"] = pd.to_numeric(
    ratings["rating"],
    errors="coerce"
)
ratings["date"] = pd.to_datetime(
    ratings["date"],
    errors="coerce"
)

# 2. Check missing values
print("Missing values before cleaning:")
print(ratings.isnull().sum())
# Remove rows with missing or invalid values
ratings = ratings.dropna(
    subset=["user_id", "movie_id", "rating", "date"]
)
# Convert numeric columns to integers after removing nulls
ratings["user_id"] = ratings["user_id"].astype("int32")
ratings["movie_id"] = ratings["movie_id"].astype("int32")
ratings["rating"] = ratings["rating"].astype("int8")

# 3. Check and remove duplicate rows
duplicate_count = ratings.duplicated().sum()
print(f"\nDuplicate rows found: {duplicate_count:,}")
if duplicate_count > 0:
    ratings = ratings.drop_duplicates()
print(f"Duplicate rows after cleaning: {ratings.duplicated().sum():,}")

# 4. Validate rating range
invalid_ratings = ~ratings["rating"].isin([1, 2, 3, 4, 5])
print(
    f"\nRows with invalid ratings: "
    f"{invalid_ratings.sum():,}"
)
# Remove invalid ratings if any exist
ratings = ratings[
    ratings["rating"].isin([1, 2, 3, 4, 5])
]

# 5. Validate movie ID range
invalid_movie_ids = ~ratings["movie_id"].between(
    1,
    17770
)
print(
    f"Rows with invalid movie IDs: "
    f"{invalid_movie_ids.sum():,}"
)
# Remove invalid movie IDs if any exist
ratings = ratings[
    ratings["movie_id"].between(1, 17770)
]

# 6. Final verification
print("\nFinal data types:")
print(ratings.dtypes)
print("\nMissing values after cleaning:")
print(ratings.isnull().sum())
print("\nRating values:")
print(sorted(ratings["rating"].unique()))
print(
    "\nMovie ID range:",
    ratings["movie_id"].min(),
    "to",
    ratings["movie_id"].max()
)
print(f"\nFinal number of rows: {len(ratings):,}")
print("\nCleaned data sample:")
print(ratings.head())

Missing values before cleaning:
user_id     0
movie_id    0
rating      0
date        0
dtype: int64

Duplicate rows found: 0
Duplicate rows after cleaning: 0

Rows with invalid ratings: 0
Rows with invalid movie IDs: 0

Final data types:
user_id              int32
movie_id             int32
rating                int8
date        datetime64[us]
dtype: object

Missing values after cleaning:
user_id     0
movie_id    0
rating      0
date        0
dtype: int64

Rating values:
[np.int8(1), np.int8(2), np.int8(3), np.int8(4), np.int8(5)]

Movie ID range: 1 to 17770

Final number of rows: 100,480,507

Cleaned data sample:
   user_id  movie_id  rating       date
0  1488844         1       3 2005-09-06
1   822109         1       5 2005-05-13
2   885013         1       4 2005-10-19
3    30878         1       4 2005-12-26
4   823519         1       3 2004-05-03


In [10]:
# STEP 4: Merge Movie Metadata

# Merge ratings with movie metadata
ratings = ratings.merge(
    movies,
    on="movie_id",
    how="left"
)

# Display merged dataset
print("Merged Dataset Shape:")
print(ratings.shape)

print("\nSample Data:")
print(ratings.head())

# Check for missing movie information
print("\nMissing Movie Metadata:")
print(ratings[["title", "year"]].isnull().sum())

# Verify the merge
print("\nUnique Movies:", ratings["movie_id"].nunique())
print("Unique Titles:", ratings["title"].nunique())

Merged Dataset Shape:
(100480507, 6)

Sample Data:
   user_id  movie_id  rating       date  year            title
0  1488844         1       3 2005-09-06  2003  Dinosaur Planet
1   822109         1       5 2005-05-13  2003  Dinosaur Planet
2   885013         1       4 2005-10-19  2003  Dinosaur Planet
3    30878         1       4 2005-12-26  2003  Dinosaur Planet
4   823519         1       3 2004-05-03  2003  Dinosaur Planet

Missing Movie Metadata:
title      0
year     965
dtype: int64

Unique Movies: 17770
Unique Titles: 17359


In [11]:
# STEP 5: Exploratory Data Analysis (EDA)

# Basic Dataset Statistics
num_users = ratings["user_id"].nunique()
num_movies = ratings["movie_id"].nunique()
num_ratings = len(ratings)
avg_rating = ratings["rating"].mean()
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Number of Users   : {num_users:,}")
print(f"Number of Movies  : {num_movies:,}")
print(f"Number of Ratings : {num_ratings:,}")
print(f"Average Rating    : {avg_rating:.2f}")

# Rating Distribution
print("\n" + "=" * 50)
print("RATING DISTRIBUTION")
print("=" * 50)
rating_distribution = (
    ratings["rating"]
    .value_counts()
    .sort_index()
)
print(rating_distribution)

# Ratings Per User
ratings_per_user = (
    ratings.groupby("user_id")
    .size()
)
print("\n" + "=" * 50)
print("RATINGS PER USER")
print("=" * 50)

print(f"Average : {ratings_per_user.mean():.2f}")
print(f"Median  : {ratings_per_user.median():.0f}")
print(f"Minimum : {ratings_per_user.min()}")
print(f"Maximum : {ratings_per_user.max()}")

# Ratings Per Movie
ratings_per_movie = (
    ratings.groupby("movie_id")
    .size()
)

print("\n" + "=" * 50)
print("RATINGS PER MOVIE")
print("=" * 50)

print(f"Average : {ratings_per_movie.mean():.2f}")
print(f"Median  : {ratings_per_movie.median():.0f}")
print(f"Minimum : {ratings_per_movie.min()}")
print(f"Maximum : {ratings_per_movie.max()}")

# Top 10 Most Rated Movies
print("\n" + "=" * 50)
print("TOP 10 MOST RATED MOVIES")
print("=" * 50)

top_movies = (
    ratings.groupby(["movie_id", "title"])
    .size()
    .reset_index(name="num_ratings")
    .sort_values("num_ratings", ascending=False)
    .head(10)
)
print(top_movies)

DATASET SUMMARY
Number of Users   : 480,189
Number of Movies  : 17,770
Number of Ratings : 100,480,507
Average Rating    : 3.60

RATING DISTRIBUTION
rating
1     4617990
2    10132080
3    28811247
4    33750958
5    23168232
Name: count, dtype: int64

RATINGS PER USER
Average : 209.25
Median  : 96
Minimum : 1
Maximum : 17653

RATINGS PER MOVIE
Average : 5654.50
Median  : 561
Minimum : 3
Maximum : 232944

TOP 10 MOST RATED MOVIES
       movie_id                                              title  \
5316       5317                                  Miss Congeniality   
15123     15124                                   Independence Day   
14312     14313                                        The Patriot   
15204     15205                             The Day After Tomorrow   
1904       1905  Pirates of the Caribbean: The Curse of the Bla...   
6286       6287                                       Pretty Woman   
11282     11283                                       Forrest Gump   
16376 

In [12]:
# STEP 6: Filter Sparse Data

MIN_USER_RATINGS = 200
MIN_MOVIE_RATINGS = 50

# Filter users
active_users = ratings.groupby("user_id").size()
active_users = active_users[active_users >= MIN_USER_RATINGS].index
ratings_filtered = ratings[
    ratings["user_id"].isin(active_users)
]

# Filter movies
popular_movies = ratings_filtered.groupby("movie_id").size()
popular_movies = popular_movies[popular_movies >= MIN_MOVIE_RATINGS].index

ratings_filtered = ratings_filtered[
    ratings_filtered["movie_id"].isin(popular_movies)
]

# Summary
print("=" * 50)
print("FILTERED DATASET SUMMARY")
print("=" * 50)
print(f"Original Ratings : {len(ratings):,}")
print(f"Filtered Ratings : {len(ratings_filtered):,}")
print(f"\nOriginal Users : {ratings['user_id'].nunique():,}")
print(f"Filtered Users : {ratings_filtered['user_id'].nunique():,}")
print(f"\nOriginal Movies : {ratings['movie_id'].nunique():,}")
print(f"Filtered Movies : {ratings_filtered['movie_id'].nunique():,}")
print(f"\nData Retained: {(len(ratings_filtered)/len(ratings))*100:.2f}%")

FILTERED DATASET SUMMARY
Original Ratings : 100,480,507
Filtered Ratings : 77,736,082

Original Users : 480,189
Filtered Users : 150,850

Original Movies : 17,770
Filtered Movies : 17,621

Data Retained: 77.36%


In [13]:

# STEP 7: Build the User–Item Sparse Matrix

from scipy.sparse import csr_matrix

# Create consecutive matrix positions for user IDs and movie IDs
user_ids = ratings_filtered["user_id"].unique()
movie_ids = ratings_filtered["movie_id"].unique()

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}
movie_to_index = {
    movie_id: index
    for index, movie_id in enumerate(movie_ids)
}
index_to_user = {
    index: user_id
    for user_id, index in user_to_index.items()
}
index_to_movie = {
    index: movie_id
    for movie_id, index in movie_to_index.items()
}

# Convert IDs into row and column positions
user_indices = ratings_filtered["user_id"].map(user_to_index)
movie_indices = ratings_filtered["movie_id"].map(movie_to_index)

# Build sparse user-item matrix
# Rows = users
# Columns = movies
# Values = ratings
user_item_matrix = csr_matrix(
    (
        ratings_filtered["rating"].astype("float32"),
        (user_indices, movie_indices)
    ),
    shape=(len(user_ids), len(movie_ids))
)

# Verify the matrix
print("User–Item Matrix Created Successfully")
print(f"Matrix shape       : {user_item_matrix.shape}")
print(f"Number of users    : {user_item_matrix.shape[0]:,}")
print(f"Number of movies   : {user_item_matrix.shape[1]:,}")
print(f"Stored ratings     : {user_item_matrix.nnz:,}")

total_cells = (
    user_item_matrix.shape[0]
    * user_item_matrix.shape[1]
)

density = (
    user_item_matrix.nnz
    / total_cells
) * 100

sparsity = 100 - density

print(f"Matrix density     : {density:.4f}%")
print(f"Matrix sparsity    : {sparsity:.4f}%")

User–Item Matrix Created Successfully
Matrix shape       : (150850, 17621)
Number of users    : 150,850
Number of movies   : 17,621
Stored ratings     : 77,736,082
Matrix density     : 2.9245%
Matrix sparsity    : 97.0755%


In [14]:
#item-user sparce matrix

# Rows = movies
# Columns = users
item_user_matrix = user_item_matrix.T.tocsr()

print("\nItem–User Matrix Shape:")
print(item_user_matrix.shape)


Item–User Matrix Shape:
(17621, 150850)
